# Model Stats - Final V5 Output

This notebook evaluates the final model output in two ways:

- strict CLO validation from the saved V5 checkpoints, including confusion matrix, per-class scores, and per-camera breakdowns
- final submission inspection, including predicted class distribution and train-vs-test comparison

In [ ]:
from pathlib import Path
import ast
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = [
    "Lateral_lying_left",
    "Lateral_lying_right",
    "Sitting",
    "Standing",
    "Sternal_lying",
]
NUM_CLASSES = len(CLASS_NAMES)

def find_existing_path(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path.resolve()
    return None

def find_data_root():
    candidates = [
        "multiview_pig_posture_recognition",
        "./multiview_pig_posture_recognition",
        "../multiview_pig_posture_recognition",
        "../../multiview_pig_posture_recognition",
        "/datasets/multi-view-pig-posture-recognition",
        "/multi-view-pig-posture-recognition",
    ]
    data_root = find_existing_path(candidates)
    if data_root is None:
        raise FileNotFoundError("Could not locate the dataset root. Check the candidate paths in this notebook.")
    return data_root

def find_run_dir(tag):
    candidates = [
        f"runs/v5_{tag.lower()}",
        f"../runs/v5_{tag.lower()}",
        f"../Merge/runs/v5_{tag.lower()}",
        f"../../Merge/runs/v5_{tag.lower()}",
        f"../Daniel/runs/v5_{tag.lower()}",
    ]
    run_dir = find_existing_path(candidates)
    if run_dir is not None:
        return run_dir
    return None

def extract_camera(image_id):
    match = re.match(r"(pen\d+_\w+_cam\d+)", image_id)
    return match.group(1) if match else "unknown"

def build_val_transform(size):
    return T.Compose([
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

print(f"Device: {DEVICE}")

In [ ]:
TAG = "T2"
DATA_ROOT = find_data_root()
TRAIN_CSV = DATA_ROOT / f"train{1 if TAG == 'T1' else 2}.csv"
TEST_CSV = DATA_ROOT / "test.csv"
IMG_DIR = DATA_ROOT / f"train{1 if TAG == 'T1' else 2}_images"
TEST_IMG_DIR = DATA_ROOT / "test_images"

SUBMISSION_CANDIDATES = [
    Path("submission.csv"),
    Path("../Daniel/submission.csv"),
    Path("../Merge/T2_v5_C.csv"),
    Path("../Merge/T2_v5_submission.csv"),
]

RUN_DIR = find_run_dir(TAG)

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"TRAIN_CSV: {TRAIN_CSV}")
print(f"TEST_CSV:  {TEST_CSV}")
print(f"IMG_DIR:   {IMG_DIR}")
print(f"RUN_DIR:   {RUN_DIR if RUN_DIR is not None else 'not found'}")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["camera"] = train_df["image_id"].apply(extract_camera)
test_df["camera"] = test_df["image_id"].apply(extract_camera)

print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Train cameras: {sorted(train_df['camera'].unique())}")
print(f"Test cameras:  {sorted(test_df['camera'].unique())}")

if "class_id" in train_df.columns:
    print("\nTraining class distribution:")
    for class_id, class_name in enumerate(CLASS_NAMES):
        count = int((train_df["class_id"] == class_id).sum())
        print(f"  {class_id} - {class_name:<22} {count:>6}")

In [ ]:
class PigPostureDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.1):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self):
        return len(self.df)

    def _crop(self, img, bbox):
        width, height = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        pad_x, pad_y = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - pad_x))
        y1 = max(0, int(y - pad_y))
        x2 = min(width, int(x + w + pad_x))
        y2 = min(height, int(y + h + pad_y))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.img_dir / row["image_id"]
        img = Image.open(image_path).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform is not None:
            crop = self.transform(crop)
        sample_key = row["row_id"] if "row_id" in row.index else idx
        return crop, int(row["class_id"]), row["image_id"], row["camera"], sample_key

def load_checkpoint_model(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    model_name = checkpoint["model_name"]
    img_size = checkpoint.get("img_size", 384)
    model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES, img_size=img_size)
    state_dict = checkpoint["model"]
    state_dict = {key.replace("module.", ""): value for key, value in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    model.to(DEVICE).eval()
    return model, checkpoint

@torch.no_grad()
def predict_dataframe(model, df, img_dir, img_size, batch_size=32, pad_ratio=0.1):
    dataset = PigPostureDataset(df, img_dir=img_dir, transform=build_val_transform(img_size), pad_ratio=pad_ratio)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    all_probs = []
    all_targets = []
    all_images = []
    all_cameras = []
    all_keys = []

    for imgs, targets, image_ids, cameras, sample_keys in tqdm(loader, desc="Predict", leave=False):
        imgs = imgs.to(DEVICE)
        with autocast(enabled=torch.cuda.is_available()):
            logits = model(imgs)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_targets.extend(targets.numpy().tolist())
        all_images.extend(image_ids)
        all_cameras.extend(cameras)
        all_keys.extend(sample_keys if isinstance(sample_keys, list) else sample_keys.numpy().tolist())

    probs = np.vstack(all_probs)
    preds = probs.argmax(axis=1)
    out = pd.DataFrame({
        "sample_key": all_keys,
        "image_id": all_images,
        "camera": all_cameras,
        "y_true": all_targets,
        "y_pred": preds,
        "confidence": probs.max(axis=1),
    })
    for class_id, class_name in enumerate(CLASS_NAMES):
        out[f"prob_{class_id}_{class_name}"] = probs[:, class_id]
    return out

def locate_submission_file():
    for candidate in SUBMISSION_CANDIDATES:
        if candidate.exists():
            return candidate.resolve()
    return None

print("Helper functions ready.")

In [ ]:
FINAL_WEIGHTS = {
    "dinov2l": 1.0,
    "convnextv2l": 0.5,
}

checkpoint_paths = []
if RUN_DIR is not None:
    checkpoint_paths = sorted(RUN_DIR.glob("best_*_fold_*.pth"))

print(f"Found {len(checkpoint_paths)} checkpoint(s).")
for path in checkpoint_paths:
    print(f"  - {path.name}")

ensemble_frames = []
per_model_summary = []

if not checkpoint_paths:
    print("\nNo checkpoints found. The notebook can still analyze the submission file below.")
else:
    predictions_by_camera = {}
    reference_keys = {}

    for ckpt_path in checkpoint_paths:
        model, ckpt = load_checkpoint_model(ckpt_path)
        prefix = ckpt.get("arch_prefix", ckpt_path.name.split("_")[1])
        val_camera = ckpt.get("val_camera", "unknown")
        img_size = ckpt.get("img_size", 384)
        pad_ratio = ckpt.get("pad_ratio", 0.1)
        batch_size = 16 if prefix == "dinov2l" else 24

        fold_df = train_df[train_df["camera"] == val_camera].copy().reset_index(drop=True)
        pred_df = predict_dataframe(model, fold_df, IMG_DIR, img_size=img_size, batch_size=batch_size, pad_ratio=pad_ratio)
        pred_df = pred_df.sort_values("sample_key").reset_index(drop=True)

        predictions_by_camera.setdefault(val_camera, {})[prefix] = pred_df
        reference_keys.setdefault(val_camera, pred_df["sample_key"].tolist())

        per_model_summary.append({
            "checkpoint": ckpt_path.name,
            "prefix": prefix,
            "camera": val_camera,
            "checkpoint_val_f1": float(ckpt.get("val_f1", np.nan)),
            "macro_f1": float(f1_score(pred_df["y_true"], pred_df["y_pred"], average="macro", zero_division=0)),
            "balanced_acc": float(balanced_accuracy_score(pred_df["y_true"], pred_df["y_pred"])),
            "mean_confidence": float(pred_df["confidence"].mean()),
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    per_model_df = (
        pd.DataFrame(per_model_summary)
        .sort_values(["camera", "prefix"])
        .reset_index(drop=True)
    )
    display(per_model_df)

    for camera, model_map in predictions_by_camera.items():
        active_models = [(prefix, df_pred, FINAL_WEIGHTS.get(prefix, 0.0)) for prefix, df_pred in model_map.items() if FINAL_WEIGHTS.get(prefix, 0.0) > 0]
        if not active_models:
            continue

        base_keys = np.array(reference_keys[camera])
        total_weight = 0.0
        ensemble_probs = np.zeros((len(base_keys), NUM_CLASSES), dtype=np.float64)
        ensemble_truth = None
        ensemble_meta = None

        for prefix, df_pred, weight in active_models:
            df_pred = df_pred.sort_values("sample_key").reset_index(drop=True)
            if not np.array_equal(df_pred["sample_key"].to_numpy(), base_keys):
                raise ValueError(f"Sample alignment mismatch for camera={camera}, prefix={prefix}")

            prob_cols = [col for col in df_pred.columns if col.startswith("prob_")]
            if ensemble_truth is None:
                ensemble_truth = df_pred["y_true"].to_numpy()
                ensemble_meta = df_pred[["sample_key", "image_id", "camera"]].copy()
            ensemble_probs += df_pred[prob_cols].to_numpy() * weight
            total_weight += weight

        ensemble_probs /= total_weight
        ensemble_pred = ensemble_probs.argmax(axis=1)
        ensemble_conf = ensemble_probs.max(axis=1)

        camera_frame = ensemble_meta.copy()
        camera_frame["y_true"] = ensemble_truth
        camera_frame["y_pred"] = ensemble_pred
        camera_frame["confidence"] = ensemble_conf
        ensemble_frames.append(camera_frame)

    ensemble_df = pd.concat(ensemble_frames, ignore_index=True)

    overall_macro_f1 = f1_score(ensemble_df["y_true"], ensemble_df["y_pred"], average="macro", zero_division=0)
    overall_balanced_acc = balanced_accuracy_score(ensemble_df["y_true"], ensemble_df["y_pred"])
    print(f"\nEnsemble macro F1: {overall_macro_f1:.4f}")
    print(f"Ensemble balanced accuracy: {overall_balanced_acc:.4f}")

    report = classification_report(
        ensemble_df["y_true"], ensemble_df["y_pred"],
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    display(report_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]])

    cm = confusion_matrix(ensemble_df["y_true"], ensemble_df["y_pred"], labels=list(range(NUM_CLASSES)))
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0]
    )
    axes[0].set_title("Confusion Matrix (Counts)")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].tick_params(axis="y", rotation=0)

    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", cmap="YlOrRd",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1]
    )
    axes[1].set_title("Confusion Matrix (Row-Normalized)")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].tick_params(axis="y", rotation=0)

    plt.tight_layout()
    plt.show()

    per_class_plot = report_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score"]].copy()
    per_class_plot.plot(kind="bar", figsize=(14, 6), ylim=(0, 1), title="Per-Class Precision / Recall / F1")
    plt.ylabel("Score")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    camera_scores = (
        ensemble_df.groupby("camera")
        .apply(lambda frame: f1_score(frame["y_true"], frame["y_pred"], average="macro", zero_division=0))
        .reset_index(name="macro_f1")
        .sort_values("macro_f1", ascending=False)
    )
    display(camera_scores)
    camera_scores.plot(kind="bar", x="camera", y="macro_f1", legend=False, figsize=(12, 4), color="#4C78A8", title="Macro F1 by Validation Camera")
    plt.ylim(0, 1)
    plt.ylabel("Macro F1")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

    confidence_summary = ensemble_df.groupby("y_pred")["confidence"].agg(["mean", "median", "min", "max", "count"]).rename_axis("predicted_class")
    display(confidence_summary)
    ensemble_df["confidence"].hist(bins=40, color="#59A96A", edgecolor="black")
    plt.title("Ensemble Confidence Distribution")
    plt.xlabel("Max probability")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

## Final Submission Inspection

This section checks the saved submission file, compares the predicted class distribution against the training distribution, and breaks the submission down by camera when `row_id` mapping is available.

In [ ]:
submission_path = locate_submission_file()
print(f"Submission file: {submission_path if submission_path is not None else 'not found'}")

if submission_path is None:
    print("No submission file found. Put a CSV with columns row_id and class_id next to this notebook or in the expected run folders.")
else:
    submission_df = pd.read_csv(submission_path)
    print(submission_df.head())

    if "class_id" not in submission_df.columns:
        raise ValueError("The submission file must contain a class_id column.")

    predicted_counts = submission_df["class_id"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index()
    train_counts = train_df["class_id"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index()
    comparison_df = pd.DataFrame({
        "train_share": train_counts / train_counts.sum(),
        "submission_share": predicted_counts / predicted_counts.sum(),
    }, index=CLASS_NAMES)
    display(comparison_df)

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    predicted_counts.plot(kind="bar", ax=axes[0], color="#4C78A8", title="Predicted Class Distribution")
    axes[0].set_xlabel("Class")
    axes[0].set_ylabel("Count")
    axes[0].set_xticks(range(NUM_CLASSES))
    axes[0].set_xticklabels(CLASS_NAMES, rotation=25, ha="right")

    comparison_df.plot(kind="bar", ax=axes[1], title="Train vs Submission Class Share")
    axes[1].set_xlabel("Class")
    axes[1].set_ylabel("Share")
    axes[1].set_ylim(0, 1)
    axes[1].set_xticklabels(CLASS_NAMES, rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

    if "row_id" in submission_df.columns and "row_id" in test_df.columns:
        submission_merged = submission_df.merge(test_df[["row_id", "camera"]], on="row_id", how="left")
        if submission_merged["camera"].notna().any():
            camera_table = pd.crosstab(submission_merged["camera"], submission_merged["class_id"], normalize="index")
            camera_table = camera_table.reindex(columns=range(NUM_CLASSES), fill_value=0)
            camera_table.columns = CLASS_NAMES
            display(camera_table)

            plt.figure(figsize=(14, 6))
            sns.heatmap(camera_table, annot=True, fmt=".2f", cmap="Blues")
            plt.title("Submission Class Share by Camera")
            plt.xlabel("Class")
            plt.ylabel("Camera")
            plt.tight_layout()
            plt.show()
        else:
            print("Camera mapping was not available for the submission rows.")